# Setup

In [1]:
import pandas as pd

In [2]:
sp500_constituents = pd.read_parquet('https://github.com/rlp-jym/zoomcamp-sma-cohort-2026/releases/download/v0.1-data/sp500_constituents_daily.parquet')
spy = pd.read_parquet('https://github.com/rlp-jym/zoomcamp-sma-cohort-2026/releases/download/v0.1-data/SPY.parquet')

In [3]:
print(sp500_constituents.info())

sp500_constituents.head().T

<class 'pandas.DataFrame'>
RangeIndex: 1566806 entries, 0 to 1566805
Data columns (total 17 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   date               1566806 non-null  datetime64[ns]
 1   index_name         1566806 non-null  str           
 2   issuer_id          1566806 non-null  string        
 3   name               1566806 non-null  string        
 4   ticker             1344034 non-null  str           
 5   open               1316261 non-null  float64       
 6   high               1316261 non-null  float64       
 7   low                1316261 non-null  float64       
 8   close              1316261 non-null  float64       
 9   adj_close          1316261 non-null  float64       
 10  volume             1316261 non-null  float64       
 11  index_weight       1566806 non-null  float64       
 12  coverage_names     1566806 non-null  float64       
 13  coverage_weight    1566806 non-null  f

,0,1,2,3,4
date,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00
index_name,SP500,SP500,SP500,SP500,SP500
issuer_id,apple,exxon mobil,google,microsoft,johnson and johnson
name,"Apple, Inc",Exxon Mobil Corp,"Google, Inc. (Class A)",Microsoft Corp,Johnson & Johnson
ticker,AAPL,XOM,NaN,MSFT,JNJ
open,22.64143,100.529999,NaN,40.950001,101.540001
high,22.672501,100.879997,NaN,41.09,102.260002
low,22.232143,99.669998,NaN,40.68,101.010002
close,22.451786,99.940002,NaN,40.790001,102.169998
adj_close,19.656939,59.890575,NaN,34.12075,73.061264


In [4]:
print(spy.info())

spy.head().T

<class 'pandas.DataFrame'>
RangeIndex: 3136 entries, 0 to 3135
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       3136 non-null   datetime64[ms]
 1   open       3136 non-null   float64       
 2   high       3136 non-null   float64       
 3   low        3136 non-null   float64       
 4   close      3136 non-null   float64       
 5   adj_close  3136 non-null   float64       
 6   volume     3136 non-null   int64         
dtypes: datetime64[ms](1), float64(5), int64(1)
memory usage: 171.6 KB
None


,0,1,2,3,4
date,2014-06-02 00:00:00,2014-06-03 00:00:00,2014-06-04 00:00:00,2014-06-05 00:00:00,2014-06-06 00:00:00
open,192.949997,192.429993,192.470001,193.410004,194.869995
high,192.990005,192.899994,193.300003,194.649994,195.429993
low,191.970001,192.25,192.270004,192.699997,194.779999
close,192.899994,192.800003,193.190002,194.449997,195.380005
adj_close,156.585037,156.503815,156.820465,157.843201,158.598129
volume,64656000,65047000,55529000,92103000,78696000


# Processing

## Index (SPY)

In [5]:
spy.isnull().sum()

date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

In [6]:
import talib

spy['turnover_value'] = (spy.close * spy.volume).astype('int64')

# change 5/20/60 proxy for weekly, monthly, quarterly
for n, tf in [(1, '1d'), (5, '5d'), (20, '20d'), (60, '60d')]:
    spy[f'chg_{tf}'] = spy.close.pct_change(n) * 100
    spy[f'chg_{tf}_max'] = ((spy.high.rolling(n).max().shift(-n) / spy.close) - 1) * 100
    spy[f'chg_{tf}_min'] = ((spy.low.rolling(n).min().shift(-n) / spy.close) - 1) * 100

spy[spy.close.shift(60).notna()].head().T

,60,61,62,63,64
date,2014-08-26 00:00:00,2014-08-27 00:00:00,2014-08-28 00:00:00,2014-08-29 00:00:00,2014-09-02 00:00:00
open,200.330002,200.429993,199.589996,200.449997,200.970001
high,200.820007,200.570007,200.270004,200.729996,201.0
low,200.279999,199.940002,199.389999,199.820007,199.860001
close,200.330002,200.25,200.139999,200.710007,200.610001
adj_close,163.395477,163.330215,163.240479,163.705444,163.623871
volume,47298000,47874000,58330000,65907000,72426000
turnover_value,9475208426,9586768500,11674166164,13228194412,14529379904
chg_1d,0.064938,-0.039935,-0.054932,0.284804,-0.049826
chg_1d_max,0.119805,0.00999,0.294792,0.144484,0.398785


In [7]:
ta_tf = 14
spy[f'atr_{ta_tf}d'] = talib.ATR(spy.high.values, spy.low.values, spy.close.values, timeperiod=ta_tf)
spy[f'rsi_{ta_tf}d'] = talib.RSI(spy.close.values, timeperiod=ta_tf)

spy.tail().T

,3131,3132,3133,3134,3135
date,2026-11-11 00:00:00,2026-11-12 00:00:00,2026-11-13 00:00:00,2026-11-16 00:00:00,2026-11-17 00:00:00
open,756.799988,753.97998,760.070007,749.659973,738.390015
high,762.51001,763.039978,760.119995,749.820007,739.76001
low,754.900024,753.669983,747.880005,738.070007,727.0
close,759.809998,757.119995,754.369995,739.950012,732.440002
adj_close,759.809998,757.119995,754.369995,739.950012,732.440002
volume,39151000,37460600,38720000,69207300,71408000
turnover_value,29747321214,28362169289,29209206210,51209942479,52302075694
chg_1d,1.318809,-0.354036,-0.363219,-1.911527,-1.014935
chg_1d_max,0.425104,0.396238,-0.603151,-0.025678,NaN


## Constituents

Null ticker and ohlc values - cannot match because of missing CUSIP before 2019 (list issuer names only); and names that got delisted or renamed has no lookup entry and stays unresolved.
- 15 rows cleared on purpose. Ticker reuse (IR, CB, AVGO) and a share-class clash (UAA Class C) failed price reconciliation; the ticker is blanked and the row kept (see config/exclusions.csv).
- These rows stay in the table because they carry index weight; removing them would shrink the denominator. coverage_names / coverage_weight measure the shortfall, and weighted_ok / equal_weight_ok gate the readings.

In [8]:
sp500_constituents.isnull().sum()

date                      0
index_name                0
issuer_id                 0
name                      0
ticker               222772
open                 250545
high                 250545
low                  250545
close                250545
adj_close            250545
volume               250545
index_weight              0
coverage_names            0
coverage_weight           0
weighted_ok               0
equal_weight_ok           0
days_since_filing         0
dtype: int64

In [16]:
list(sp500_constituents.name[sp500_constituents.ticker.isnull()].unique())

['google,_inc._(class_a)',
 'cvs_caremark_corp',
 'biogen_idec,_inc',
 'e._i._du_pont_de_nemours_&_co',
 'priceline.com,_inc',
 'twenty-first_century_fox,_inc._(class_a)',
 'hewlett-packard_co',
 'monsanto_co',
 'the_dow_chemical_co',
 'express_scripts_holding_co',
 'time_warner,_inc',
 'walgreen_co',
 'emc_corp',
 'anadarko_petroleum_corp',
 'dominion_resources,_inc',
 'praxair,_inc',
 'time_warner_cable,_inc',
 'allergan,_inc',
 'precision_castparts_corp',
 'directv',
 'actavis_plc',
 'cbs_corp._(class_b)',
 'freeport-mcmoran_copper_&_gold,_inc',
 'yahoo!,_inc',
 'ace,_ltd',
 'kraft_foods_group,_inc',
 'national-oilwell_varco,_inc',
 'covidien_plc',
 'apache_corp',
 'johnson_controls,_inc',
 'raytheon_co',
 'bb&t_corp',
 'wellpoint,_inc',
 'aetna,_inc',
 'discover_financial_services',
 'pioneer_natural_resources_co',
 'noble_energy,_inc',
 'spectra_energy_corp',
 'marathon_oil_corp',
 'forest_laboratories,_inc',
 'suntrust_banks,_inc',
 'lorillard,_inc',
 'tyco_international,_ltd',
 

In [10]:
for c in list(sp500_constituents.select_dtypes('str').columns):
    sp500_constituents[c] = sp500_constituents[c].str.lower().str.replace(' ', '_')    

print(sp500_constituents.info())

sp500_constituents.head().T

<class 'pandas.DataFrame'>
RangeIndex: 1566806 entries, 0 to 1566805
Data columns (total 17 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   date               1566806 non-null  datetime64[ns]
 1   index_name         1566806 non-null  str           
 2   issuer_id          1566806 non-null  string        
 3   name               1566806 non-null  string        
 4   ticker             1344034 non-null  str           
 5   open               1316261 non-null  float64       
 6   high               1316261 non-null  float64       
 7   low                1316261 non-null  float64       
 8   close              1316261 non-null  float64       
 9   adj_close          1316261 non-null  float64       
 10  volume             1316261 non-null  float64       
 11  index_weight       1566806 non-null  float64       
 12  coverage_names     1566806 non-null  float64       
 13  coverage_weight    1566806 non-null  f

,0,1,2,3,4
date,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00,2014-06-02 00:00:00
index_name,sp500,sp500,sp500,sp500,sp500
issuer_id,apple,exxon_mobil,google,microsoft,johnson_and_johnson
name,"apple,_inc",exxon_mobil_corp,"google,_inc._(class_a)",microsoft_corp,johnson_&_johnson
ticker,aapl,xom,NaN,msft,jnj
open,22.64143,100.529999,NaN,40.950001,101.540001
high,22.672501,100.879997,NaN,41.09,102.260002
low,22.232143,99.669998,NaN,40.68,101.010002
close,22.451786,99.940002,NaN,40.790001,102.169998
adj_close,19.656939,59.890575,NaN,34.12075,73.061264


In [ ]:
daily[daily.ticker.isnull()]

In [ ]:
spy.head()